In [1]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import defaultdict

CURRENT_STYLES = [
    ("letter",     "Just say the letter: 'A' or 'B'. Nothing else."),
    ("ordinal",    "Say 'the first one' if they picked A, or 'the second one' if they picked B. Nothing else."),
    ("positional", "Say 'the left one' if they picked A, or 'the right one' if they picked B. Nothing else."),
    ("direct",     "Commit clearly to their choice in one short sentence. The output MUST contain the letter (A or B) or a position word (left, right, first, second). Examples: 'Option A.', 'I'll go with B.', 'A for sure.'"),
    ("casual",     "Casual and short, 3-6 words. The output MUST contain the letter (A or B) or a position word (left, right, first, second). Never emit bare positive words alone. Examples: 'A works', 'B for me', 'gimme A', 'yeah A', 'B looks good'."),
]

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)

free_gpu()
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
verbalizer = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", dtype=torch.float16, device_map="cuda",
)
verbalizer.eval()
print(f"verbalizer loaded, VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def verbalize(picked_letter, style_name, style_prompt, temperature=0.8, top_p=0.9):
    system = (
        f"You are a user searching for an image. You have been shown two options, "
        f"A (first, left) and B (second, right). You have decided you prefer option "
        f"{picked_letter}. Respond in the following style:\n\n{style_prompt}\n\n"
        f"Reply with ONLY the utterance, no explanation, no preamble."
    )
    messages = [{"role": "system", "content": system}]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = verbalizer.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()


def has_referent(utterance):
    u = utterance.lower()
    if " a " in f" {u} " or " b " in f" {u} " or u.startswith(("a ", "b ")) or u in ("a", "b"):
        return True
    if "option a" in u or "option b" in u or "'a'" in u or "'b'" in u:
        return True
    position_words = ["left", "right", "first", "second", "1st", "2nd", "#1", "#2", "one", "two"]
    if any(w in u.split() for w in position_words):
        return True
    return False


N_SAMPLES = 5
results = []
for style_name, style_prompt in CURRENT_STYLES:
    print(f"\n--- style: {style_name} ---")
    n_with_ref = 0
    n_total = 0
    for picked in ["A", "B"]:
        for _ in range(N_SAMPLES):
            utt = verbalize(picked, style_name, style_prompt)
            has_ref = has_referent(utt)
            n_total += 1
            if has_ref:
                n_with_ref += 1
            mark = "OK  " if has_ref else "MISS"
            print(f"  [{mark}] picked={picked}  utt={utt!r}")
            results.append((style_name, picked, utt, has_ref))
    print(f"  -> {n_with_ref}/{n_total} contain a referent")

print(f"\n\n=== VERBALIZER AUDIT SUMMARY ===")
per_style = defaultdict(lambda: [0, 0])
for style, picked, utt, has_ref in results:
    per_style[style][1] += 1
    if has_ref:
        per_style[style][0] += 1

for style, (n_ok, n_total) in per_style.items():
    print(f"  {style:12s} {n_ok}/{n_total} = {100*n_ok/n_total:.0f}% contain referent")

del verbalizer, tok
free_gpu()

VRAM: 0.00 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

verbalizer loaded, VRAM: 3.09 GB

--- style: letter ---
  [OK  ] picked=A  utt='A'
  [OK  ] picked=A  utt='A'
  [OK  ] picked=A  utt='A'
  [OK  ] picked=A  utt='A'
  [OK  ] picked=A  utt='A'
  [OK  ] picked=B  utt='B'
  [OK  ] picked=B  utt='B'
  [OK  ] picked=B  utt='B'
  [OK  ] picked=B  utt='B'
  [OK  ] picked=B  utt='B'
  -> 10/10 contain a referent

--- style: ordinal ---
  [OK  ] picked=A  utt='the second one'
  [OK  ] picked=A  utt='the first one'
  [OK  ] picked=A  utt='the second one'
  [OK  ] picked=A  utt='the second one'
  [OK  ] picked=A  utt='the second one'
  [OK  ] picked=B  utt='the second one'
  [OK  ] picked=B  utt='the second one'
  [OK  ] picked=B  utt='the second one'
  [OK  ] picked=B  utt='the second one'
  [OK  ] picked=B  utt='the second one'
  -> 10/10 contain a referent

--- style: positional ---
  [OK  ] picked=A  utt='the left one'
  [OK  ] picked=A  utt='the left one'
  [OK  ] picked=A  utt='the left one'
  [OK  ] picked=A  utt='the left one'
  [OK  ] pic

In [2]:
import re

# Rule-based ground-truth classifier for verbalizer utterances.
# Returns "A", "B", "AMBIGUOUS" (contains both/neither), or "UNCLEAR" (no signal).
# This is what a perfect parser SHOULD output.
def ground_truth_meaning(utterance):
    u = utterance.lower().strip().rstrip(".!?,")

    # explicit position + ordinal words
    LEFT_A = ["left", "first", "1st", "#1"]
    RIGHT_B = ["right", "second", "2nd", "#2"]

    has_a_letter = bool(re.search(r"\ba\b", u))
    has_b_letter = bool(re.search(r"\bb\b", u))
    has_left = any(w in u.split() for w in LEFT_A)
    has_right = any(w in u.split() for w in RIGHT_B)

    a_signal = has_a_letter or has_left
    b_signal = has_b_letter or has_right

    if a_signal and b_signal:
        return "AMBIGUOUS"
    if a_signal:
        return "A"
    if b_signal:
        return "B"
    return "UNCLEAR"


# Sanity check on the ground-truth classifier before we trust it
for utt, expected in [
    ("A", "A"), ("B", "B"),
    ("the first one", "A"), ("the second one", "B"),
    ("the left one", "A"), ("the right one", "B"),
    ("Option A.", "A"), ("Option B.", "B"),
    ("I'll go with B.", "B"), ("A for sure", "A"),
    ("gimme A", "A"), ("B for me", "B"),
    ("A fits well, smooth.", "A"),
    ("B fits better", "B"),
    ("Sure thing B", "B"),
    ("Sure, B fits better here.", "B"),
    ("B looks good, thanks!", "B"),
    ("I prefer A - it's got character.", "A"),
    ("works great", "UNCLEAR"),
]:
    got = ground_truth_meaning(utt)
    mark = "OK  " if got == expected else "FAIL"
    print(f"  [{mark}] {utt!r:40s} expected={expected}  got={got}")

  [OK  ] 'A'                                      expected=A  got=A
  [OK  ] 'B'                                      expected=B  got=B
  [OK  ] 'the first one'                          expected=A  got=A
  [OK  ] 'the second one'                         expected=B  got=B
  [OK  ] 'the left one'                           expected=A  got=A
  [OK  ] 'the right one'                          expected=B  got=B
  [OK  ] 'Option A.'                              expected=A  got=A
  [OK  ] 'Option B.'                              expected=B  got=B
  [OK  ] "I'll go with B."                        expected=B  got=B
  [OK  ] 'A for sure'                             expected=A  got=A
  [OK  ] 'gimme A'                                expected=A  got=A
  [OK  ] 'B for me'                               expected=B  got=B
  [OK  ] 'A fits well, smooth.'                   expected=A  got=A
  [OK  ] 'B fits better'                          expected=B  got=B
  [OK  ] 'Sure thing B'                         

In [3]:
import gc, json, torch
from pathlib import Path
from collections import defaultdict, Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
CHECKPOINTS = PROJECT / "checkpoints"
PARSER_ADAPTER = CHECKPOINTS / "qwen3b-qlora-v2" / "checkpoint-final"

# Current shipped verbalizer prompts from config.py
CURRENT_STYLES = [
    ("letter",     "Just say the letter: 'A' or 'B'. Nothing else."),
    ("ordinal",    "Say 'the first one' if they picked A, or 'the second one' if they picked B. Nothing else."),
    ("positional", "Say 'the left one' if they picked A, or 'the right one' if they picked B. Nothing else."),
    ("direct",     "Commit clearly to their choice in one short sentence. The output MUST contain the letter (A or B) or a position word (left, right, first, second). Examples: 'Option A.', 'I'll go with B.', 'A for sure.'"),
    ("casual",     "Casual and short, 3-6 words. The output MUST contain the letter (A or B) or a position word (left, right, first, second). Never emit bare positive words alone. Examples: 'A works', 'B for me', 'gimme A', 'yeah A', 'B looks good'."),
]

SYSTEM_PROMPT_V3 = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough. To count as an endorsement, the user must reference the options - either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)


import re
def ground_truth_meaning(utterance):
    u = utterance.lower().strip().rstrip(".!?,")
    LEFT_A = ["left", "first", "1st", "#1"]
    RIGHT_B = ["right", "second", "2nd", "#2"]
    has_a_letter = bool(re.search(r"\ba\b", u))
    has_b_letter = bool(re.search(r"\bb\b", u))
    has_left = any(w in u.split() for w in LEFT_A)
    has_right = any(w in u.split() for w in RIGHT_B)
    a_signal = has_a_letter or has_left
    b_signal = has_b_letter or has_right
    if a_signal and b_signal:
        return "AMBIGUOUS"
    if a_signal:
        return "A"
    if b_signal:
        return "B"
    return "UNCLEAR"


free_gpu()

# ---------- Load verbalizer (1.5B fp16) ----------
print("loading verbalizer (1.5B fp16)...")
verb_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
if verb_tok.pad_token is None:
    verb_tok.pad_token = verb_tok.eos_token
verbalizer = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", dtype=torch.float16, device_map="cuda",
)
verbalizer.eval()
print(f"VRAM after verbalizer: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def verbalize(picked_letter, style_prompt, rng_seed):
    system = (
        f"You are a user searching for an image. You have been shown two options, "
        f"A (first, left) and B (second, right). You have decided you prefer option "
        f"{picked_letter}. Respond in the following style:\n\n{style_prompt}\n\n"
        f"Reply with ONLY the utterance, no explanation, no preamble."
    )
    messages = [{"role": "system", "content": system}]
    inputs = verb_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    torch.manual_seed(rng_seed)
    with torch.no_grad():
        out = verbalizer.generate(
            **inputs, max_new_tokens=15, do_sample=True,
            temperature=0.8, top_p=0.9, pad_token_id=verb_tok.eos_token_id,
        )
    return verb_tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()


# ---------- Generate all verbalizer outputs first ----------
N_SAMPLES = 20
print(f"\ngenerating {N_SAMPLES} samples per (style, picked) pair...")
generations = []
for style_name, style_prompt in CURRENT_STYLES:
    for picked in ["A", "B"]:
        for i in range(N_SAMPLES):
            utt = verbalize(picked, style_prompt, rng_seed=hash((style_name, picked, i)) & 0xFFFFFFFF)
            gt = ground_truth_meaning(utt)
            generations.append({
                "style": style_name, "picked": picked, "utt": utt, "gt": gt,
            })
print(f"generated {len(generations)} utterances")

# unload verbalizer to free VRAM before loading parser
del verbalizer, verb_tok
free_gpu()

# ---------- Load parser (3B v2 + LoRA) ----------
print("\nloading parser (3B v2)...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
)
parser_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
if parser_tok.pad_token is None:
    parser_tok.pad_token = parser_tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", quantization_config=bnb, device_map="cuda",
)
parser = PeftModel.from_pretrained(base, str(PARSER_ADAPTER))
parser.eval()
print(f"VRAM after parser: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def parse_utt(utterance):
    """Returns parsed_meaning: 'A', 'B', 'UNCLEAR', 'AMBIGUOUS', or 'MALFORMED'."""
    user_msg = f'Options: ["A", "B"]\nUser: {utterance}'
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V3},
        {"role": "user", "content": user_msg},
    ]
    inputs = parser_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = parser.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=parser_tok.eos_token_id,
        )
    raw = parser_tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return "MALFORMED"
    if parsed == "*":
        return "UNCLEAR"
    if parsed == []:
        return "UNCLEAR"
    if isinstance(parsed, list):
        if set(parsed) == {"A"}:
            return "A"
        if set(parsed) == {"B"}:
            return "B"
        if set(parsed) == {"A", "B"}:
            return "AMBIGUOUS"
    return "MALFORMED"


# ---------- Parse everything ----------
print(f"\nparsing all {len(generations)} utterances...")
for i, g in enumerate(generations):
    if i % 50 == 0:
        print(f"  {i}/{len(generations)}", flush=True)
    g["parsed"] = parse_utt(g["utt"])
print("done.")


# ---------- Analyze ----------
print(f"\n{'='*90}")
print(f"FLIP-SOURCE DECOMPOSITION per style")
print(f"{'='*90}\n")

# For each style, compute:
#   - verbalizer flip = ground_truth_meaning != picked (or unclear/ambiguous)
#   - parser flip     = parsed != ground_truth_meaning (only measurable when gt is A or B)
#   - roundtrip flip  = parsed != picked (or unclear/ambiguous)
per_style = defaultdict(lambda: {
    "n": 0,
    "verb_flip": 0, "verb_unclear": 0, "verb_ambiguous": 0, "verb_ok": 0,
    "parser_flip_when_gt_clear": 0, "parser_unclear_when_gt_clear": 0, "parser_ok_when_gt_clear": 0,
    "n_gt_clear": 0,
    "roundtrip_flip": 0, "roundtrip_unclear": 0, "roundtrip_ambiguous": 0, "roundtrip_ok": 0,
})

for g in generations:
    s = per_style[g["style"]]
    s["n"] += 1

    # verbalizer side
    if g["gt"] == g["picked"]:
        s["verb_ok"] += 1
    elif g["gt"] in ("A", "B"):
        s["verb_flip"] += 1
    elif g["gt"] == "UNCLEAR":
        s["verb_unclear"] += 1
    elif g["gt"] == "AMBIGUOUS":
        s["verb_ambiguous"] += 1

    # parser side (only measurable when verbalizer gave a clear signal)
    if g["gt"] in ("A", "B"):
        s["n_gt_clear"] += 1
        if g["parsed"] == g["gt"]:
            s["parser_ok_when_gt_clear"] += 1
        elif g["parsed"] in ("A", "B"):
            s["parser_flip_when_gt_clear"] += 1
        else:  # UNCLEAR/AMBIGUOUS/MALFORMED
            s["parser_unclear_when_gt_clear"] += 1

    # roundtrip
    if g["parsed"] == g["picked"]:
        s["roundtrip_ok"] += 1
    elif g["parsed"] in ("A", "B"):
        s["roundtrip_flip"] += 1
    elif g["parsed"] == "AMBIGUOUS":
        s["roundtrip_ambiguous"] += 1
    else:  # UNCLEAR/MALFORMED
        s["roundtrip_unclear"] += 1

# print per style
print(f"{'style':<12s} | {'verb_ok%':<9s} {'verb_flip%':<11s} {'verb_unclear%':<14s} | {'parser_ok%*':<12s} {'parser_flip%*':<14s} | {'roundtrip_ok%':<14s} {'roundtrip_flip%':<16s}")
print(f"{'':12s} | {'':9s} {'(wrong opt)':<11s} {'(no signal)':<14s} | {'when clear':<12s} {'when clear':<14s} | {'':14s} {'':16s}")
print("-" * 130)

totals = defaultdict(int)
for style_name in ["letter", "ordinal", "positional", "direct", "casual"]:
    s = per_style[style_name]
    n = s["n"]
    ncl = s["n_gt_clear"]
    print(f"{style_name:<12s} | "
          f"{100*s['verb_ok']/n:>6.1f}%   {100*s['verb_flip']/n:>7.1f}%     {100*s['verb_unclear']/n:>7.1f}%        | "
          f"{100*s['parser_ok_when_gt_clear']/max(1,ncl):>7.1f}%     {100*s['parser_flip_when_gt_clear']/max(1,ncl):>7.1f}%       | "
          f"{100*s['roundtrip_ok']/n:>7.1f}%       {100*s['roundtrip_flip']/n:>7.1f}%")
    for k, v in s.items():
        totals[k] += v

n = totals["n"]
ncl = totals["n_gt_clear"]
print("-" * 130)
print(f"{'TOTAL':<12s} | "
      f"{100*totals['verb_ok']/n:>6.1f}%   {100*totals['verb_flip']/n:>7.1f}%     {100*totals['verb_unclear']/n:>7.1f}%        | "
      f"{100*totals['parser_ok_when_gt_clear']/max(1,ncl):>7.1f}%     {100*totals['parser_flip_when_gt_clear']/max(1,ncl):>7.1f}%       | "
      f"{100*totals['roundtrip_ok']/n:>7.1f}%       {100*totals['roundtrip_flip']/n:>7.1f}%")

# show some flip examples
print(f"\n\n=== VERBALIZER FLIP EXAMPLES (oracle picked X, verbalizer said Y) ===")
flip_examples = defaultdict(list)
for g in generations:
    if g["gt"] in ("A", "B") and g["gt"] != g["picked"]:
        flip_examples[g["style"]].append(g)
for style, exs in flip_examples.items():
    print(f"\n{style}:")
    for g in exs[:5]:
        print(f"  picked={g['picked']}  utt={g['utt']!r}  gt={g['gt']}  parsed={g['parsed']}")

del parser, base, parser_tok
free_gpu()

VRAM: 0.01 GB
loading verbalizer (1.5B fp16)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM after verbalizer: 3.10 GB

generating 20 samples per (style, picked) pair...
generated 200 utterances
VRAM: 0.01 GB

loading parser (3B v2)...


W0903 20:04:30.178000 25608 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM after parser: 2.20 GB

parsing all 200 utterances...
  0/200


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  50/200
  100/200
  150/200
done.

FLIP-SOURCE DECOMPOSITION per style

style        | verb_ok%  verb_flip%  verb_unclear%  | parser_ok%*  parser_flip%*  | roundtrip_ok%  roundtrip_flip% 
             |           (wrong opt) (no signal)    | when clear   when clear     |                                
----------------------------------------------------------------------------------------------------------------------------------
letter       |  100.0%       0.0%         0.0%        |   100.0%         0.0%       |   100.0%           0.0%
ordinal      |   65.0%      35.0%         0.0%        |   100.0%         0.0%       |    65.0%          35.0%
positional   |  100.0%       0.0%         0.0%        |   100.0%         0.0%       |   100.0%           0.0%
direct       |   72.5%       0.0%        27.5%        |   100.0%         0.0%       |    72.5%           0.0%
casual       |   90.0%       7.5%         0.0%        |   100.0%         0.0%       |    92.5%           7.5%
--------------

In [4]:
import gc, json, torch, sys, re
from pathlib import Path
from collections import defaultdict
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
CHECKPOINTS = PROJECT / "checkpoints"
PARSER_ADAPTER = CHECKPOINTS / "qwen3b-qlora-v2" / "checkpoint-final"

# Make the comparison-search-app package importable so we can use its real
# VerbalizerLLM (which now has the validation gate).
APP_ROOT = Path(r"C:\Users\shlok\projects\ddp-llm\comparison-search-app-merged")
sys.path.insert(0, str(APP_ROOT))

# Force-reload in case an older version is cached from a previous notebook run
for mod in list(sys.modules):
    if mod.startswith("app."):
        del sys.modules[mod]

from app import config  # noqa: E402
from app.verbalizer_llm import VerbalizerLLM  # noqa: E402
from app.parser_llm import ParserLLM  # noqa: E402


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)


def ground_truth_meaning(utterance):
    u = utterance.lower().strip().rstrip(".!?,")
    LEFT_A = ["left", "first", "1st", "#1"]
    RIGHT_B = ["right", "second", "2nd", "#2"]
    has_a_letter = bool(re.search(r"\ba\b", u))
    has_b_letter = bool(re.search(r"\bb\b", u))
    has_left = any(w in u.split() for w in LEFT_A)
    has_right = any(w in u.split() for w in RIGHT_B)
    a_signal = has_a_letter or has_left
    b_signal = has_b_letter or has_right
    if a_signal and b_signal:
        return "AMBIGUOUS"
    if a_signal:
        return "A"
    if b_signal:
        return "B"
    return "UNCLEAR"


free_gpu()

# ---------- Load parser first (needed to pass into verbalizer) ----------
print("loading parser (3B v2)...")
parser = ParserLLM()
print(f"VRAM after parser: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\nloading verbalizer (1.5B)...")
verbalizer = VerbalizerLLM()
print(f"VRAM after verbalizer: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


# ---------- Generate + parse WITH validation gate ----------
N_SAMPLES = 20
print(f"\ngenerating {N_SAMPLES} samples per (style, picked) pair WITH validation gate...")

# Force the verbalizer to visit each style equally by monkey-patching its style choice.
# We do this by looping over styles manually and calling _generate_one directly,
# but keeping the retry logic from verbalize() intact via a wrapper.
generations = []
for style_idx, (style_name, style_instruction) in enumerate(verbalizer.styles):
    for picked in ["A", "B"]:
        for i in range(N_SAMPLES):
            rng = np.random.default_rng(hash((style_name, picked, i)) & 0xFFFFFFFF)

            # Manually mimic what verbalize() does but pin the style_name we
            # want to test. We temporarily replace styles so the RNG picks it.
            orig_styles = verbalizer.styles
            verbalizer.styles = [(style_name, style_instruction)]
            try:
                utt, style_returned = verbalizer.verbalize(picked, rng, parser=parser)
            finally:
                verbalizer.styles = orig_styles

            gt = ground_truth_meaning(utt)
            parsed, _ = parser.parse(utt, options=("A", "B"))
            if parsed is None:
                parsed_meaning = "MALFORMED"
            elif parsed == "*" or parsed == []:
                parsed_meaning = "UNCLEAR"
            elif isinstance(parsed, list):
                if set(parsed) == {"A"}:
                    parsed_meaning = "A"
                elif set(parsed) == {"B"}:
                    parsed_meaning = "B"
                elif set(parsed) == {"A", "B"}:
                    parsed_meaning = "AMBIGUOUS"
                else:
                    parsed_meaning = "MALFORMED"
            else:
                parsed_meaning = "MALFORMED"

            generations.append({
                "style": style_name,
                "style_returned": style_returned,
                "picked": picked,
                "utt": utt,
                "gt": gt,
                "parsed": parsed_meaning,
                "fell_back": style_returned == "letter (fallback)",
            })

print(f"generated {len(generations)} utterances")


# ---------- Analyze ----------
print(f"\n{'='*100}")
print(f"WITH VALIDATION GATE — decomposition per style")
print(f"{'='*100}\n")

per_style = defaultdict(lambda: {"n": 0, "roundtrip_ok": 0, "roundtrip_flip": 0,
                                   "roundtrip_unclear": 0, "fell_back": 0})
for g in generations:
    s = per_style[g["style"]]
    s["n"] += 1
    if g["fell_back"]:
        s["fell_back"] += 1
    if g["parsed"] == g["picked"]:
        s["roundtrip_ok"] += 1
    elif g["parsed"] in ("A", "B"):
        s["roundtrip_flip"] += 1
    else:
        s["roundtrip_unclear"] += 1

print(f"{'style':<12s} | {'roundtrip_ok%':<15s} {'flip%':<10s} {'unclear%':<11s} | {'fell_back%':<12s}")
print("-" * 70)
totals = defaultdict(int)
for style_name in ["letter", "ordinal", "positional", "direct", "casual"]:
    s = per_style[style_name]
    n = s["n"]
    print(f"{style_name:<12s} | {100*s['roundtrip_ok']/n:>8.1f}%       {100*s['roundtrip_flip']/n:>6.1f}%    {100*s['roundtrip_unclear']/n:>6.1f}%      | {100*s['fell_back']/n:>6.1f}%")
    for k, v in s.items():
        totals[k] += v
print("-" * 70)
n = totals["n"]
print(f"{'TOTAL':<12s} | {100*totals['roundtrip_ok']/n:>8.1f}%       {100*totals['roundtrip_flip']/n:>6.1f}%    {100*totals['roundtrip_unclear']/n:>6.1f}%      | {100*totals['fell_back']/n:>6.1f}%")


# ---------- Compare against last time (no gate) ----------
print(f"\n\n=== BEFORE vs AFTER validation gate ===")
before = {"letter": 100, "ordinal": 65, "positional": 100, "direct": 72.5, "casual": 92.5, "TOTAL": 86.0}
print(f"{'style':<12s} | {'before':<10s} {'after':<10s} {'delta':<10s}")
print("-" * 50)
for style_name in ["letter", "ordinal", "positional", "direct", "casual"]:
    s = per_style[style_name]
    n = s["n"]
    after = 100 * s['roundtrip_ok'] / n
    delta = after - before[style_name]
    print(f"{style_name:<12s} | {before[style_name]:>6.1f}%    {after:>6.1f}%    {delta:+6.1f}pp")
n = totals["n"]
after = 100 * totals['roundtrip_ok'] / n
delta = after - before["TOTAL"]
print("-" * 50)
print(f"{'TOTAL':<12s} | {before['TOTAL']:>6.1f}%    {after:>6.1f}%    {delta:+6.1f}pp")


# ---------- Show any remaining failures ----------
remaining = [g for g in generations if g["parsed"] != g["picked"]]
if remaining:
    print(f"\n\n=== REMAINING FAILURES ({len(remaining)}) ===")
    for g in remaining[:10]:
        print(f"  style={g['style']:12s}  picked={g['picked']}  utt={g['utt']!r:50s}  parsed={g['parsed']}  fell_back={g['fell_back']}")

del verbalizer, parser
free_gpu()

VRAM: 0.01 GB
loading parser (3B v2)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM after parser: 2.20 GB

loading verbalizer (1.5B)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM after verbalizer: 5.28 GB

generating 20 samples per (style, picked) pair WITH validation gate...
generated 200 utterances

WITH VALIDATION GATE — decomposition per style

style        | roundtrip_ok%   flip%      unclear%    | fell_back%  
----------------------------------------------------------------------
letter       |    100.0%          0.0%       0.0%      |    0.0%
ordinal      |    100.0%          0.0%       0.0%      |    0.0%
positional   |    100.0%          0.0%       0.0%      |    0.0%
direct       |    100.0%          0.0%       0.0%      |    2.5%
casual       |    100.0%          0.0%       0.0%      |    5.0%
----------------------------------------------------------------------
TOTAL        |    100.0%          0.0%       0.0%      |    1.5%


=== BEFORE vs AFTER validation gate ===
style        | before     after      delta     
--------------------------------------------------
letter       |  100.0%     100.0%      +0.0pp
ordinal      |   65.0%     100.0%  